# SpaceX API collection and documented frozen-data fallback

**AI-assisted research coursework for Amir Saulebekov.** The live API timed out in both the Mac and official lab. This notebook records a fresh probe, normalizes the official API JSON snapshot, and explicitly joins the official enriched CSV. The join replaces unavailable per-rocket, payload, core and launchpad API calls. It does not claim a successful fresh API collection or a reconstruction of missing raw payload masses.

## Task 1: Request and normalize launch records

In [1]:
from pathlib import Path
import datetime
import json
import requests
import numpy as np
import pandas as pd
url='https://api.spacexdata.com/v4/launches/past'
probe={}
try:
    response=requests.get(url,timeout=8)
    probe={'url':url,'status':response.status_code,'bytes':len(response.content)}
except requests.RequestException as exc:
    probe={'url':url,'error_type':type(exc).__name__,'error':str(exc)}
print(json.dumps(probe,indent=2))
Path('api-probe.json').write_text(json.dumps(probe,indent=2))
raw=json.loads(Path('../API_call_spacex_api.json').read_text())
data=pd.json_normalize(raw)
print('Official JSON snapshot records:',len(data))
display(data[['flight_number','date_utc','rocket','payloads','cores','launchpad']].head())

{
  "url": "https://api.spacexdata.com/v4/launches/past",
  "error_type": "ReadTimeout",
  "error": "HTTPSConnectionPool(host='api.spacexdata.com', port=443): Read timed out. (read timeout=8)"
}
Official JSON snapshot records: 107


,flight_number,date_utc,rocket,payloads,cores,launchpad
0,1,2006-03-24T22:30:00.000Z,5e9d0d95eda69955f709d1eb,[5eb0e4b5b6c3bb0006eeb1e1],"[{'core': '5e9e289df35918033d3b2623', 'flight'...",5e9e4502f5090995de566f86
1,2,2007-03-21T01:10:00.000Z,5e9d0d95eda69955f709d1eb,[5eb0e4b6b6c3bb0006eeb1e2],"[{'core': '5e9e289ef35918416a3b2624', 'flight'...",5e9e4502f5090995de566f86
2,3,2008-08-03T03:34:00.000Z,5e9d0d95eda69955f709d1eb,"[5eb0e4b6b6c3bb0006eeb1e3, 5eb0e4b6b6c3bb0006e...","[{'core': '5e9e289ef3591814873b2625', 'flight'...",5e9e4502f5090995de566f86
3,4,2008-09-28T23:15:00.000Z,5e9d0d95eda69955f709d1eb,[5eb0e4b7b6c3bb0006eeb1e5],"[{'core': '5e9e289ef3591855dc3b2626', 'flight'...",5e9e4502f5090995de566f86
4,5,2009-07-13T03:35:00.000Z,5e9d0d95eda69955f709d1eb,[5eb0e4b7b6c3bb0006eeb1e6],"[{'core': '5e9e289ef359184f103b2627', 'flight'...",5e9e4502f5090995de566f86


In [2]:
data=data[['rocket','payloads','launchpad','cores','flight_number','date_utc']].copy()
data=data.loc[data['cores'].map(len).eq(1) & data['payloads'].map(len).eq(1)].copy()
data['cores']=data['cores'].map(lambda value:value[0])
data['payloads']=data['payloads'].map(lambda value:value[0])
data['Date']=pd.to_datetime(data['date_utc']).dt.strftime('%Y-%m-%d')
data=data.loc[data['Date'].le('2020-11-13')].copy()
print('Single-core, single-payload records before Falcon 9 filter:',len(data))
display(data.head())

Single-core, single-payload records before Falcon 9 filter: 94


,rocket,payloads,launchpad,cores,flight_number,date_utc,Date
0,5e9d0d95eda69955f709d1eb,5eb0e4b5b6c3bb0006eeb1e1,5e9e4502f5090995de566f86,"{'core': '5e9e289df35918033d3b2623', 'flight':...",1,2006-03-24T22:30:00.000Z,2006-03-24
1,5e9d0d95eda69955f709d1eb,5eb0e4b6b6c3bb0006eeb1e2,5e9e4502f5090995de566f86,"{'core': '5e9e289ef35918416a3b2624', 'flight':...",2,2007-03-21T01:10:00.000Z,2007-03-21
3,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e5,5e9e4502f5090995de566f86,"{'core': '5e9e289ef3591855dc3b2626', 'flight':...",4,2008-09-28T23:15:00.000Z,2008-09-28
4,5e9d0d95eda69955f709d1eb,5eb0e4b7b6c3bb0006eeb1e6,5e9e4502f5090995de566f86,"{'core': '5e9e289ef359184f103b2627', 'flight':...",5,2009-07-13T03:35:00.000Z,2009-07-13
5,5e9d0d95eda69973a809d1ec,5eb0e4b7b6c3bb0006eeb1e7,5e9e4501f509094ba4566f84,"{'core': '5e9e289ef359185f2b3b2628', 'flight':...",6,2010-06-04T18:45:00.000Z,2010-06-04


## Task 2: Filter to Falcon 9 and construct the enriched dataframe
The official CSV supplies the Falcon 9 names and enrichment fields. A one-to-one date join links it to actual raw JSON records. The five Falcon 1 records do not occur in the Falcon 9 CSV. The API and SQL datasets have different populations.

In [3]:
enriched=pd.read_csv('../dataset_part_1.csv')
assert enriched['Date'].is_unique and data['Date'].is_unique
raw_columns=['Date','rocket','launchpad','payloads','cores','flight_number']
linked=data[raw_columns].merge(enriched,on='Date',how='inner',validate='one_to_one')
assert len(linked)==90 and linked['BoosterVersion'].eq('Falcon 9').all()
falcon9_rocket_ids=set(linked['rocket'])
data['BoosterVersion']=data['rocket'].map(lambda value:'Falcon 9' if value in falcon9_rocket_ids else 'Excluded: no match in Falcon 9 snapshot')
print(data['BoosterVersion'].value_counts())
data_falcon9=linked[enriched.columns].copy()
data_falcon9.loc[:,'FlightNumber']=range(1,len(data_falcon9)+1)
display(data_falcon9.head())

BoosterVersion
Falcon 9                                   90
Excluded: no match in Falcon 9 snapshot     4
Name: count, dtype: int64


,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6104.959412,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCAFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## Task 3: Deal with missing payload values
The supplied enriched CSV already contains imputed payload mass. The requested mean/fill code runs below. No missing raw values are fabricated to make the exercise appear successful.

In [4]:
before=int(data_falcon9['PayloadMass'].isna().sum())
payload_mean=data_falcon9['PayloadMass'].mean()
data_falcon9['PayloadMass']=data_falcon9['PayloadMass'].replace(np.nan,payload_mean)
print({'payload_missing_before':before,'payload_mean_kg':payload_mean,
       'payload_missing_after':int(data_falcon9['PayloadMass'].isna().sum())})
display(data_falcon9.isna().sum())
data_falcon9.to_csv('dataset_part_1_fallback.csv',index=False)

{'payload_missing_before': 0, 'payload_mean_kg': np.float64(6104.959411764707), 'payload_missing_after': 0}


FlightNumber       0
Date               0
BoosterVersion     0
PayloadMass        0
Orbit              0
LaunchSite         0
Outcome            0
Flights            0
GridFins           0
Reused             0
Legs               0
LandingPad        26
Block              0
ReusedCount        0
Serial             0
Longitude          0
Latitude           0
dtype: int64

## Sources and limits
Official JSON: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/API_call_spacex_api.json

Official enrichment: https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_1.csv

The frozen dates end in November 2020. A working live service would be needed to rerun the original enrichment calls and evaluate the original missing-payload distribution. The subsequent labs explicitly allow the provided datasets.